In [ ]:
import sys
import os

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'parameters'))
sys.path.insert(0, os.getcwd())

from analysis_utils import (
    load_best_runs, load_all_runs, extract_history, add_wall_times,
    plot_convergence_curves, plot_speedrun_results,
    plot_best_of_budget,
    print_ranking_table, print_variant_comparison,
    print_variant_comparison_allruns, print_offdiag_ranking,
    bootstrap_best_run_ci, print_topk_comparison, print_robustness_table,
    print_best_hyperparameters, save_best_hyperparameters,
    get_optimizer_colors, ema,
    TASK_CONFIGS,
)

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

BACKEND = "local"
RESULTS_DIR = os.path.join('..', '..', 'results')

In [ ]:
def load_task(task_key):
    """Load all data for a task. Returns (cfg, best_runs, all_runs, smoothed_data, raw_data, colors)."""
    cfg = TASK_CONFIGS[task_key]
    colors = get_optimizer_colors(cfg['optimizers'])

    best_runs = load_best_runs(
        backend=BACKEND, optimizers=cfg['optimizers'],
        task_tag=cfg['task_tag'], results_dir=RESULTS_DIR,
        metric_key=cfg['metric_key'], direction=cfg['direction'],
        sort_metric=cfg['sort_metric'], sort_order=cfg['sort_order'],
        iteration=cfg['iteration'],
    )
    all_runs = load_all_runs(
        backend=BACKEND, optimizers=cfg['optimizers'],
        task_tag=cfg['task_tag'], results_dir=RESULTS_DIR,
        iteration=cfg['iteration'],
    )
    raw_data = extract_history(BACKEND, best_runs, cfg['history_metrics'])
    add_wall_times(raw_data)

    smoothed = {}
    for opt, data in raw_data.items():
        d = dict(data)
        for key, fn in cfg.get('y_transforms', {}).items():
            d[key] = fn(d)
        for key in cfg['y_keys']:
            if key in d and len(d[key]) > 0:
                d[key] = ema(np.asarray(d[key], dtype=float))
        smoothed[opt] = d

    return cfg, best_runs, all_runs, smoothed, raw_data, colors


def show_results(cfg, best_runs, all_runs, smoothed, raw_data, colors):
    """Plot all results for a single task."""
    name = cfg['display_name']
    itr = cfg['iteration']

    # --- Training curves ---
    fig = plot_convergence_curves(
        smoothed, y_keys=cfg['y_keys'], x_keys=['epoch', 'wall_times'],
        mark_best=False, colors=colors, highlight=cfg.get('highlight'),
        y_labels=cfg.get('y_labels'),
        x_labels={'epoch': 'Epoch', 'wall_times': 'Wall Time (s)'},
        log_y=cfg.get('log_y', 'auto'),
    )
    for ax in fig.axes:
        for yk, lims in cfg.get('ylim', {}).items():
            if ax.get_ylabel() == cfg.get('y_labels', {}).get(yk, yk):
                ax.set_ylim(*lims)
    plt.tight_layout()
    plt.show()

    # --- Speedrun ---
    fig = plot_speedrun_results(
        raw_data, cfg['speedrun_targets'],
        metric_key=cfg['speedrun_metric'],
        direction=cfg['speedrun_direction'], colors=colors,
    )
    plt.show()

    # --- Rankings & statistics ---
    print_ranking_table(best_runs, metric_key=cfg['metric_key'],
                        direction=cfg['direction'],
                        title=f'{name} (itr {itr}) — Optimizer Ranking')
    print_variant_comparison(best_runs, metric_key=cfg['metric_key'],
                             direction=cfg['direction'],
                             title=f'{name} (itr {itr}) — Variant Pairs (Best Run)')
    print_variant_comparison_allruns(all_runs, metric_key=cfg['metric_key'],
                                     direction=cfg['direction'],
                                     title=f'{name} (itr {itr}) — Variant Comparison (All Runs)')
    print_offdiag_ranking(best_runs, metric_key=cfg['metric_key'],
                          direction=cfg['direction'],
                          title=f'{name} (itr {itr}) — Off-Diagonal (a,b) Ranking')
    bootstrap_best_run_ci(all_runs, metric_key=cfg['metric_key'],
                          direction=cfg['direction'],
                          title=f'{name} (itr {itr}) — Bootstrap 95% CI')
    print_topk_comparison(all_runs, metric_key=cfg['metric_key'],
                          direction=cfg['direction'],
                          title=f'{name} (itr {itr}) — Top-k Mean Comparison')
    print_robustness_table(all_runs, metric_key=cfg['metric_key'],
                           direction=cfg['direction'],
                           title=f'{name} (itr {itr}) — Robustness Metrics')

    # --- Best-of-budget ---
    fig = plot_best_of_budget(
        all_runs, metric_key=cfg['metric_key'],
        direction=cfg['direction'], colors=colors,
        title=f'{name} (itr {itr}) — Best-of-Budget',
    )
    plt.show()

    # --- HP export ---
    print_best_hyperparameters(best_runs)
    filename = f"best_hyperparameters_{cfg['task_tag']}_itr_{itr}.csv"
    save_best_hyperparameters(best_runs, filename)

# MNIST MLP

In [ ]:
mnist = load_task("mnist_mlp")
show_results(*mnist)

# CIFAR-10 ResNet-18

In [ ]:
cifar = load_task("cifar10_resnet18")
show_results(*cifar)

# Shakespeare MiniGPT

In [ ]:
shake = load_task("shakespeare_minigpt")
show_results(*shake)

# Regression

In [ ]:
reg = load_task("regression")
show_results(*reg)